In [ ]:
from __future__ import annotations

"""Lean mined factorlib submission: vol_adjusted_pullback.

This is a compact export of the wide miner's best candidate. It intentionally
computes only the fields required by the expression to reduce submission runtime
and memory pressure.
"""

import numpy as np
import pandas as pd


FACTORLIB_COLUMNS = [
    "date",
    "instrument",
    "close",
    "volume",
    "amount",
    "daily_return",
    "macd_hist_12_26_9",
    "netflow_amount_rate_main",
    "net_active_buy_amount_main",
]


def import_dai():
    try:
        import dai  # type: ignore

        return dai
    except Exception:
        from bigquant import dai  # type: ignore

        return dai


def _numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        df[col] = df.groupby("date", observed=True)[col].transform(
            lambda s: s.fillna(s.median()) if s.notna().any() else s.fillna(0.0)
        )
        df[col] = df[col].fillna(0.0)
    return df


def _rolling(df: pd.DataFrame, column: str, window: int, min_periods: int, func: str = "mean") -> pd.Series:
    grouped = df.groupby("instrument", observed=True)[column]
    if func == "sum":
        return grouped.transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    if func == "std":
        return grouped.transform(lambda s: s.rolling(window, min_periods=min_periods).std())
    return grouped.transform(lambda s: s.rolling(window, min_periods=min_periods).mean())


def _rank(df: pd.DataFrame, values: pd.Series, ascending: bool = True) -> pd.Series:
    values = pd.Series(values, index=df.index)
    values = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)
    ranked = values.groupby(df["date"], observed=True).rank(pct=True, ascending=ascending)
    return ranked.fillna(0.5) - 0.5


def _prod(a: pd.Series, b: pd.Series) -> pd.Series:
    gate = (b.fillna(0.0) + 0.5).clip(0.0, 1.0)
    return a.fillna(0.0) * gate


def fetch_panel(dai, query_start: str, query_end: str) -> pd.DataFrame:
    columns_sql = ",\n        ".join(FACTORLIB_COLUMNS)
    sql = f"""
    SELECT
        {columns_sql}
    FROM bigalpha_2026_factorlib
    ORDER BY date, instrument
    """
    panel = dai.query(sql, filters={"date": [query_start, query_end]}, compression=True).df()
    panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
    panel["instrument"] = panel["instrument"].astype(str)
    return panel


def _build_factor(panel: pd.DataFrame) -> pd.DataFrame:
    df = panel.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df = _numeric(df, [c for c in FACTORLIB_COLUMNS if c not in {"date", "instrument"}])

    df["ret_3_sum"] = _rolling(df, "daily_return", 3, 2, "sum")
    df["ret_5_sum"] = _rolling(df, "daily_return", 5, 3, "sum")
    df["ret_20_sum"] = _rolling(df, "daily_return", 20, 10, "sum")
    df["ret_60_sum"] = _rolling(df, "daily_return", 60, 30, "sum")
    df["vol_20"] = _rolling(df, "daily_return", 20, 10, "std")
    df["flow_3"] = _rolling(df, "netflow_amount_rate_main", 3, 2)
    df["flow_5"] = _rolling(df, "netflow_amount_rate_main", 5, 3)
    df["active_buy_5"] = _rolling(df, "net_active_buy_amount_main", 5, 3)
    df["active_buy_20"] = _rolling(df, "net_active_buy_amount_main", 20, 10)
    df["vwap_proxy"] = np.where(df["volume"] > 0, df["amount"] / df["volume"], np.nan)
    df["vwap_to_close"] = np.where(df["close"] > 0, df["vwap_proxy"] / df["close"] - 1.0, np.nan)
    df = _numeric(
        df,
        [
            "ret_3_sum",
            "ret_5_sum",
            "ret_20_sum",
            "ret_60_sum",
            "vol_20",
            "flow_3",
            "flow_5",
            "active_buy_5",
            "active_buy_20",
            "vwap_to_close",
        ],
    )

    low_vol20 = _rank(df, -df["vol_20"])
    voladj_pullback5 = _rank(df, -np.where(df["vol_20"] > 1e-8, df["ret_5_sum"] / df["vol_20"], np.nan))
    voladj_pullback3 = _rank(df, -np.where(df["vol_20"] > 1e-8, df["ret_3_sum"] / df["vol_20"], np.nan))
    mom60_20 = _rank(df, df["ret_60_sum"] - df["ret_20_sum"])
    mom20_5 = _rank(df, df["ret_20_sum"] - df["ret_5_sum"])
    vwap_discount = _rank(df, -df["vwap_to_close"])
    flow_in3 = _rank(df, df["flow_3"])
    flow_in5 = _rank(df, df["flow_5"])
    active_buy = _rank(df, df["active_buy_5"] + df["active_buy_20"])
    macd_up = _rank(df, df["macd_hist_12_26_9"])

    flow_momentum = (
        0.25 * flow_in3
        + 0.25 * flow_in5
        + 0.20 * active_buy
        + 0.15 * mom20_5
        + 0.15 * macd_up
    )
    vol_adjusted_pullback = (
        0.34 * voladj_pullback5
        + 0.24 * _prod(mom60_20, voladj_pullback3)
        + 0.20 * low_vol20
        + 0.22 * vwap_discount
    )
    factor = (
        0.46 * vol_adjusted_pullback
        + 0.20 * _prod(mom60_20, voladj_pullback5)
        + 0.14 * vwap_discount
        + 0.12 * low_vol20
        + 0.08 * flow_momentum
    )

    out = df.loc[:, ["date", "instrument"]].copy()
    out["factor"] = factor.astype("float64").replace([np.inf, -np.inf], np.nan)
    out["factor"] = out.groupby("date", observed=True)["factor"].transform(lambda s: s.fillna(s.median()))
    out["factor"] = out["factor"].fillna(0.0)
    out["factor"] = _rank(out, out["factor"])
    return out.loc[:, ["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main(
    datasources: dict | str | None = None,
    start_date: str = "2019-01-01 00:00:00",
    end_date: str = "2025-12-31 23:59:59",
) -> pd.DataFrame:
    dai = import_dai()
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=90)).strftime("%Y-%m-%d %H:%M:%S")
    panel = fetch_panel(dai, query_start, end_date)
    out = _build_factor(panel)
    start_ts = pd.to_datetime(start_date).normalize()
    end_ts = pd.to_datetime(end_date).normalize()
    return out[(out["date"] >= start_ts) & (out["date"] <= end_ts)].reset_index(drop=True)
